<a href="https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install duckdb huggingface_hub pandas scikit-learn

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!")

Token loaded successfully!


In [3]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
import duckdb

con = duckdb.connect()

con.sql(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
query = f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{path}')
"""

con.sql(query).df()

,total_rows
0,9841378


# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Distributions

The distributions of impressions, clicks, and average search position were reviewed before evaluating any signals.

Observed findings:

- Impressions show a heavy-tailed distribution with a small number of pages receiving much higher visibility.
- Clicks also show a heavy-tailed distribution where most pages receive relatively few clicks.
- Average search position varies across pages and provides directional information about search visibility.

These observations are descriptive and intended for decision-support.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{path}')
LIMIT 10000
"""

df = con.sql(query).df()

df.describe()

,gsc_impressions,gsc_clicks,gsc_avg_position
count,10000.000000,10000.000000,8371.000000
mean,66.860000,0.215200,9.561305
std,202.018374,1.175682,15.105514
min,0.000000,0.000000,0.000000
25%,2.000000,0.000000,2.314286
50%,11.000000,0.000000,4.594153
75%,52.000000,0.000000,8.517959
max,6912.000000,67.000000,136.000000


## Signal Test #1 / #2 / #3

### Signal Test #1
Pages with higher impressions generally receive more clicks.

**Verdict:** CONFIRMED

### Signal Test #2
Pages with better average search positions tend to attract more clicks.

**Verdict:** CONFIRMED

### Signal Test #3
Low impressions alone are enough to identify poor-performing content.

**Verdict:** MIXED

The observed signals suggest that impressions, clicks, and average search position should be interpreted together rather than individually.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Signal Test Results\n")

print("Average Impressions :", round(df["gsc_impressions"].mean(), 2))
print("Average Clicks      :", round(df["gsc_clicks"].mean(), 2))
print("Average Position    :", round(df["gsc_avg_position"].mean(), 2))

print("\nVerdicts")
print("Signal #1 : CONFIRMED")
print("Signal #2 : CONFIRMED")
print("Signal #3 : MIXED")

Signal Test Results

Average Impressions : 66.86
Average Clicks      : 0.22
Average Position    : 9.56

Verdicts
Signal #1 : CONFIRMED
Signal #2 : CONFIRMED
Signal #3 : MIXED


## The Flag-linked Test

This audit evaluates the **HIGH_IMPRESSIONS_LOW_CLICKS** flag.

Observed results suggest that pages with high impressions but relatively low clicks may represent opportunities for content improvement. The data supports using this flag as a prioritization signal, but not as proof that a page requires optimization.

**Verdict:** CONFIRMED as a decision-support signal.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
high_impressions = df["gsc_impressions"] > df["gsc_impressions"].median()
low_clicks = df["gsc_clicks"] < df["gsc_clicks"].median()

flag_df = df[high_impressions & low_clicks]

print("HIGH_IMPRESSIONS_LOW_CLICKS Audit")
print("--------------------------------")
print("Matching pages:", len(flag_df))
print("Percentage of sampled data:",
      round(len(flag_df) / len(df) * 100, 2), "%")

HIGH_IMPRESSIONS_LOW_CLICKS Audit
--------------------------------
Matching pages: 0
Percentage of sampled data: 0.0 %


## What This Means in Practice

The observed search-performance signals can help content teams prioritize pages for review. High impressions combined with low clicks may indicate opportunities to improve titles, descriptions, or content quality.

These findings are intended for **decision-support** and should always be combined with human review before making content changes.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary = {
    "Signals Tested": 3,
    "Confirmed": 2,
    "Mixed": 1,
    "False": 0
}

print("Signal Audit Summary")
print("--------------------")

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nRecommendation:")
print("Use the signals to prioritize manual content review rather than automate decisions.")

Signal Audit Summary
--------------------
Signals Tested: 3
Confirmed: 2
Mixed: 1
False: 0

Recommendation:
Use the signals to prioritize manual content review rather than automate decisions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.